# Poisson-GLM

This notebook implements Generalized Linear Model (GLM) analysis inspired by the neuroGLM toolkit (https://github.com/memming/neuroGLM) and the methodology from:

**Park et al. (2014). "Encoding and decoding in parietal cortex during sensorimotor decision-making." Nature Neuroscience 17, 1395-1403.**



In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import time
from scipy.io import loadmat
from scipy import stats, signal
from scipy.sparse import csr_matrix, vstack, hstack, issparse
from scipy.optimize import minimize
from scipy.special import gammaln
from scipy.ndimage import gaussian_filter1d
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, r2_score
from pathlib import Path
import os
import warnings
warnings.filterwarnings('ignore')


# Set style for publication-quality plots
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")


In [ ]:
from imports import *
from src.utils import poisson_glm_utils
from config import dir_config

from config.poisson_glm_config import StateBasedPoissonGLMConfig

In [ ]:
compiled_dir = Path(dir_config.data.compiled)
processed_dir = Path(dir_config.data.processed)
output_folder_name = 'equal_block_cross_validation_1coh_50choice'

session_to_exclude = ["210210_GP_JP", "241209_GP_TZ"]

session_metadata = pd.read_csv(Path(processed_dir, 'sessions_metadata.csv'))
session_metadata = session_metadata[~np.isin(session_metadata["session_id"], session_to_exclude)]

neuron_metadata = pd.read_csv(Path(processed_dir, 'neuron_metadata.csv'))
neuron_metadata = neuron_metadata[~np.isin(neuron_metadata["session_id"], session_to_exclude)].reset_index()

with open(Path(processed_dir, f'glm_hmm_models', f'glm_hmm_masked_final.pkl'), 'rb') as f:
    glm_hmm = pickle.load(f)

poisson_glm_config = StateBasedPoissonGLMConfig()

In [ ]:
# Feature indices for easy access
feature_idx = {
    'target_start': 0,
    'target_end': poisson_glm_config.FEATURES_TARGET,
    'stim_start': poisson_glm_config.FEATURES_TARGET,
    'stim_end': poisson_glm_config.FEATURES_TARGET + poisson_glm_config.FEATURES_STIMULUS,
    'saccade_start': poisson_glm_config.FEATURES_TARGET + poisson_glm_config.FEATURES_STIMULUS,
    'saccade_end': poisson_glm_config.FEATURES_TARGET + poisson_glm_config.FEATURES_STIMULUS + poisson_glm_config.FEATURES_SACCADE,
    'history_start': poisson_glm_config.FEATURES_TARGET + poisson_glm_config.FEATURES_STIMULUS + poisson_glm_config.FEATURES_SACCADE,
    'history_end': poisson_glm_config.FEATURES_TARGET + poisson_glm_config.FEATURES_STIMULUS + poisson_glm_config.FEATURES_SACCADE + poisson_glm_config.FEATURES_HISTORY,
    'intercept_idx': poisson_glm_config.get_total_features() - 1
}

print(f"Total features: {poisson_glm_config.get_total_features()}")

In [ ]:
poisson_glm_config.N_COHERENCE_LEVELS == 1

### Helper Functions

In [ ]:
def extract_neuron_data(neuron_id):
    session_name = neuron_metadata.loc[neuron_metadata["neuron_id"] == neuron_id, "session_id"].values[0]
    data_path = Path(compiled_dir, session_name)

    print(f"Analyzing neuron {neuron_id} from session {session_name}")

    # Load neural and behavioral data
    try:
        spike_times = np.load(data_path / "spike_times.npy")
        spike_clusters = np.load(data_path / "spike_clusters.npy")
    except:
        spike_times = loadmat(Path(compiled_dir, session_name, "spike_times.mat"))
        spike_times = spike_times["spike_times"][0]
        spike_clusters = loadmat(Path(compiled_dir, session_name, "spike_clusters.mat"))
        spike_clusters = spike_clusters["spike_clusters"][0]

    # Get neuron spike times
    cluster_id = neuron_metadata.cluster[neuron_metadata["neuron_id"] == neuron_id].values[0]
    neuron_spike_times = spike_times[spike_clusters == cluster_id]
    neuron_spike_times = (neuron_spike_times / 30).round().astype(int)  # Convert to ms

    # Get timestamps and trial data
    timestamps = pd.read_csv(Path(compiled_dir, session_name, f"{session_name}_timestamps.csv"), index_col=None)
    trial_info = pd.read_csv(Path(compiled_dir, session_name, f"{session_name}_trial.csv"), index_col=None)

    # Process trial data
    GP_trial_data = trial_info[trial_info.task_type == 1].reset_index(drop=True)
    # signed coherence
    GP_trial_data["signed_coherence"] = GP_trial_data["coherence"] * (2*GP_trial_data["target"]-1)
    
    GP_trial_data = GP_trial_data[GP_trial_data.reaction_time.notna()]
    # include equal block only
    GP_trial_data = GP_trial_data[GP_trial_data.prob_toRF == 50]
    GP_trial_data["state"] = 0

    coh_levels = np.sort(GP_trial_data['signed_coherence'].unique()) /100  # Normalize coherence

    # Keep only correct trials
    GP_trial_data = GP_trial_data[GP_trial_data.outcome == 1].reset_index()

    # print(f"Found {len(GP_trial_data)} valid trials")
    # print(f"Neuron has {len(neuron_spike_times)} spikes")
    # print(f"State distribution: {GP_trial_data.state.value_counts().to_dict()}")

    return GP_trial_data, neuron_spike_times, timestamps, coh_levels

def create_neuroglm_trials(session_data, timestamps, neuron_spike_times, bin_size=1.0):
    """
    Create trial structure following neuroGLM format.
    """
    trials = []

    # Convert timestamps to ms
    timestamps_ms = (timestamps / 30).round()

    for idx, row in session_data.iterrows():
        trial_idx = row.trial_number - 1  # Convert to 0-based

        # Trial timing (relative to target onset - 50ms)
        target_onset = timestamps_ms.loc[trial_idx, "target_onset"]
        trial_start = target_onset - 50
        trial_end = timestamps_ms.loc[trial_idx, "response_onset"]
        duration = trial_end - trial_start # -50ms of target onset to response onset

        if pd.isna(duration) or duration <= 0:
            continue

        # Get trial spike times (relative to trial start)
        trial_spikes = neuron_spike_times[
            (neuron_spike_times >= trial_start) &
            (neuron_spike_times <= trial_end)
        ] - trial_start

        # Create binned spike train
        n_bins = int(np.ceil(duration / bin_size))
        spike_train = np.zeros(n_bins)

        for spike_time in trial_spikes:
            bin_idx = int(np.floor(spike_time / bin_size))
            if 0 <= bin_idx < n_bins:
                spike_train[bin_idx] += 1

        # Event timings (relative to trial start)
        events = {
            'target_onset': 50,  # Always 50ms into trial
            'stimulus_onset': timestamps_ms.loc[trial_idx, "stimulus_onset"] - trial_start,
            'stimulus_offset': timestamps_ms.loc[trial_idx, "response_onset"] - trial_start,
            'response_onset': timestamps_ms.loc[trial_idx, "response_onset"] - trial_start,
        }

        # Trial structure
        trial = {
            'duration': duration,
            'spike_train': spike_train,
            'n_bins': n_bins,

            # Event timings
            'target_onset': events['target_onset'],
            'stimulus_onset': events['stimulus_onset'],
            'stimulus_offset': events['stimulus_offset'],
            'response_onset': events['response_onset'],

            # Experimental variables
            'coherence': row.signed_coherence / 100,  # Normalize coherence
            'choice': row.choice,
            'state': row.state,  # Bias state from GLM-HMM
            'reaction_time': row.reaction_time,

            # Trial metadata
            'trial_idx': int(trial_idx),
            # 'original_idx': idx
        }

        trials.append(trial)

    return trials

def build_design_matrix(trials, coh_levels):
    # Initialize containers
    trial_matrices = []
    trial_spike_trains = []

    # Process each trial
    for trial in trials:
        trial_duration = int(trial['duration'])
        trial_design = np.zeros((trial_duration, poisson_glm_config.get_total_features()))

        # 1. TARGET ONSET COMPONENT
        target_bin = int(trial['target_onset'])
        if 0 < target_bin <= trial_duration:
            target_matrix = np.zeros((trial_duration, 1))
            target_matrix[target_bin-1] = 1.0
            target_conv, _ = poisson_glm_utils.convolve_with_basis(
                target_matrix,
                poisson_glm_config.TARGET_BASIS,
                poisson_glm_config.TARGET_DURATION_MS,
                poisson_glm_config.TARGET_SPACING_MS,
                effect= poisson_glm_config.TARGET_EFFECT
            )

            target_start = feature_idx['target_start'] + int(trial['state']) * poisson_glm_config.TARGET_N_BASES
            target_end = target_start + poisson_glm_config.TARGET_N_BASES
            trial_design[:, target_start:target_end] = target_conv


        # 2. STIMULUS COHERENCE COMPONENT
        stim_bin = int(trial['stimulus_onset'])
        resp_bin = int(trial['response_onset'])
        if 0 < stim_bin < resp_bin <= trial_duration:
            stim_matrix = np.zeros((trial_duration, 1))
            stim_matrix[stim_bin-1:resp_bin] = 1.0
            stim_conv, _ = poisson_glm_utils.convolve_with_basis(
                stim_matrix,
                poisson_glm_config.STIMULUS_BASIS,
                poisson_glm_config.STIMULUS_DURATION_MS,
                poisson_glm_config.STIMULUS_SPACING_MS,
                effect= poisson_glm_config.STIMULUS_EFFECT
            )

            # Assign to coherence-specific and state-specific features

            
            if poisson_glm_config.N_COHERENCE_LEVELS == 1:
                coh_idx = 0 # same stimulus kernel for all coherence levels (1stimulus)
            else:
                coh_idx = np.where(coh_levels == trial['coherence'])[0][0]
            state_idx = int(trial['state'])
            coh_start = feature_idx['stim_start'] + coh_idx * poisson_glm_config.STIMULUS_N_BASES + state_idx * poisson_glm_config.STIMULUS_N_BASES * len(coh_levels)
            coh_end = coh_start + poisson_glm_config.STIMULUS_N_BASES
            trial_design[:, coh_start:coh_end] = stim_conv

        # 3. SACCADE/CHOICE COMPONENT
        if 0 < resp_bin <= trial_duration:
            saccade_matrix = np.zeros((trial_duration, 1))
            saccade_matrix[resp_bin-1] = 1.0
            saccade_conv, _ = poisson_glm_utils.convolve_with_basis(
                saccade_matrix,
                poisson_glm_config.SACCADE_BASIS,
                poisson_glm_config.SACCADE_DURATION_MS,
                poisson_glm_config.SACCADE_SPACING_MS,
                effect= poisson_glm_config.SACCADE_EFFECT
            )

            # Assign to choice-specific features
            choice_idx = int(trial['choice'])
            state_idx = int(trial['state'])
            choice_start = feature_idx['saccade_start'] + choice_idx * poisson_glm_config.SACCADE_N_BASES + state_idx * poisson_glm_config.SACCADE_N_BASES * poisson_glm_config.N_CHOICE_OPTIONS
            choice_end = choice_start + poisson_glm_config.SACCADE_N_BASES
            trial_design[:, choice_start:choice_end] = saccade_conv


        # 4. POST-SPIKE HISTORY COMPONENT
        history_matrix = poisson_glm_utils.create_post_spike_history_matrix(trial['spike_train'])
        trial_design[:, feature_idx['history_start']:feature_idx['history_end']] = history_matrix

        # 5. INTERCEPT TERM
        trial_design[:, feature_idx['intercept_idx']] = 1.0

        # Store processed trial
        trial_matrices.append(csr_matrix(trial_design))
        trial_spike_trains.append(trial['spike_train'])

    X = vstack(trial_matrices, format='csr')
    y = np.concatenate(trial_spike_trains)

    return X, y

def fit_poisson_glm(X, y):
    if issparse(X):
        X = X.toarray()#.astype(np.float16)  # Convert to dense for faster computation in this case
    # Strategy 2: Feature standardization for better conditioning
    X_means = X.mean(axis=0)
    X_stds = X.std(axis=0)

    # Avoid division by zero
    X_stds[X_stds < 1e-8] = 1.0

    # Standardize all features except intercept
    X_scaled = X.copy()
    X_scaled[:, :-1] = (X[:, :-1] - X_means[:-1]) / X_stds[:-1]

    def loss_fun(w):
        eta = X_scaled @ w
        if np.any(y[eta < -15]>0):
            return 1e20  # Penalty if rate is 0 but spikes are present
        else:
            eta = np.clip(eta, -15, 15)  # Prevent overflow
            mu = np.exp(eta)
            return np.sum(mu) - np.dot(y, eta) + 0.1 * np.dot(w, w)

    def grad_fun(w):
        eta = X_scaled @ w
        eta = np.clip(eta, -15, 15)
        mu = np.exp(eta)
        return X_scaled.T @ (mu - y) + 0.02 * w

    n_features = X_scaled.shape[1]
    w_init = np.zeros(n_features)
    w_init[-1] = np.log(max(y.mean(), 1e-8))  # Smart intercept

    # print(f"\nOptimizing {n_features} parameters...")
    start_time = time.time()

    result = minimize(
        fun=loss_fun,
        x0=w_init,
        method='L-BFGS-B',
        jac=grad_fun,
        options={
            'maxiter': 1000,    # Very limited iterations
            'gtol': 1e-3,      # Relaxed tolerance
            'ftol': 1e-5,      # Relaxed tolerance
            'maxfun': 200      # Limit function calls
        }
    )

    fit_time = time.time() - start_time
    # print(f"Optimization completed in {fit_time:.1f} seconds")

    if result.success or result.fun < 1e6:
        weights_scaled = result.x

        # don't save for memory
        # predicted y
        eta = X_scaled @ weights_scaled
        predicted = np.exp(np.clip(eta, -15, 15))
        result['predicted_y'] = predicted
        # result['X_scaled'] = X_scaled

        # result["mean_squared_error"] = np.mean((y - predicted)**2)
        # Transform weights back to original scale
        # fitted_weights = weights_scaled.copy()
        # fitted_weights[:-1] = weights_scaled[:-1] / X_stds[:-1]
        # fitted_weights[-1] = weights_scaled[-1] - np.dot(weights_scaled[:-1], X_means[:-1] / X_stds[:-1])
        # result["fitted_weights"] = fitted_weights
    else:
        print(f"Optimization failed: {result.message}")
        return None

    return result

def predict_poisson_glm(X, model):
    if issparse(X):
        X = X.toarray()#.astype(np.float16)
    X_means = X.mean(axis=0)
    X_stds = X.std(axis=0)

    # Avoid division by zero
    X_stds[X_stds < 1e-8] = 1.0

    X_scaled = X.copy()
    X_scaled[:, :-1] = (X[:, :-1] - X_means[:-1]) / X_stds[:-1]

    eta = X_scaled @ model.x
    return np.exp(np.clip(eta, -15, 15))


## 1. Load and Prepare Data

### save config for verification

In [ ]:
config_path = Path(processed_dir, 'poisson_glm', output_folder_name, 'config.json')
import os

os.makedirs(config_path.parent, exist_ok=True)
poisson_glm_config.save(config_path)

In [ ]:
cfg = StateBasedPoissonGLMConfig.load(config_path)

In [ ]:
cfg.get_total_features()

### Cross-validation

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=216)


for neuron_id in neuron_metadata.neuron_id.unique():
    # save each neuron fitting result separately
    output_path = Path(processed_dir, 'poisson_glm', output_folder_name, f'{neuron_id}.pkl')
    # os.makedirs(output_path.parent, exist_ok=True)
    if not output_path.exists():
        print(f"Processing Neuron {neuron_id}")
        session_data, neuron_spike_times, timestamps, coh_levels = extract_neuron_data(neuron_id)
        fitting_result = {
            "coh_levels": coh_levels,
            "folds": []
        }

        for fold, (train_idx, test_idx) in enumerate(kf.split(session_data)):
            # -----------------------
            # TRAIN
            # -----------------------
            print(f"Fold {fold+1} / 5")
            train_data = session_data.iloc[train_idx]
            train_trials = create_neuroglm_trials(
                train_data, timestamps, neuron_spike_times,
                poisson_glm_config.BIN_SIZE_MS
            )

            X_train, y_train = build_design_matrix(train_trials, coh_levels)
            model = fit_poisson_glm(X_train, y_train)

            # unpack train predictions per trial
            train_preds, onset = [], 0
            for t in train_trials:
                n = t["n_bins"]
                train_preds.append(model["predicted_y"][onset:onset+n])
                onset += n

            # -----------------------
            # TEST
            # -----------------------
            test_data = session_data.iloc[test_idx]
            test_trials = create_neuroglm_trials(
                test_data, timestamps, neuron_spike_times,
                poisson_glm_config.BIN_SIZE_MS
            )

            X_test, y_test = build_design_matrix(test_trials, coh_levels)
            test_flat_pred = predict_poisson_glm(X_test, model)

            test_preds, onset = [], 0
            for t in test_trials:
                n = t["n_bins"]
                test_preds.append(test_flat_pred[onset:onset+n])
                onset += n

            # -----------------------
            # STORE
            # -----------------------
            fitting_result["folds"].append({
                "fold": fold,
                "train_data": pd.DataFrame(train_trials),
                "test_data": pd.DataFrame(test_trials),
                "model": model,
                "train_predictions": train_preds,
                "test_predictions": test_preds
            })

        
        with open(output_path, 'wb') as f:
            pickle.dump(fitting_result, f)
    else:
        print(f"Neuron {neuron_id} already processed.")
    